In [188]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

In [189]:
l_c_d = pd.read_csv('/content/survey lung cancer.csv')
#l_c_d => lung cancer detection

In [203]:
X = l_c_d.drop('LUNG_CANCER', axis=1)
y = l_c_d['LUNG_CANCER']#<-- target column


# Convert 'GENDER' column to numerical using one-hot encoding
X = pd.get_dummies(X, columns=['GENDER'], drop_first=True)


In [201]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state = 67)
                                                                #tests with 20% of the data and trains wtih the remaining 80%

In [202]:
gbch = GradientBoostingClassifier(n_estimators=1000, max_depth=3, learning_rate=0.01, random_state=67)
rfch = RandomForestClassifier(n_estimators=1000, random_state=67)
#^ the h is for hyperparameter

stacking = StackingClassifier(
    estimators=[('gb', gbc), ('rf', rfc)],

     # vv meta learner
    final_estimator = LogisticRegression(class_weight='balanced',
                                       C=60,
                                       solver="lbfgs",
                                       max_iter=1000
                                       ),

    # uses cross-val(cv) predictions to train meta-learner
    cv = 4


)

stacking.fit(X_train, y_train)
print(f"GBRF Accuracy = {stacking.score(X_test, y_test):.3f}")
        #^ gradient boosted random forest
H = stacking.score(X_test, y_test)

GBRF Accuracy = 0.952


In [193]:
H

0.9516129032258065

In [194]:
#Control

gbcc = GradientBoostingClassifier()
rfcc = RandomForestClassifier()
#^ the extra c means control

stacking = StackingClassifier(

    estimators=[('gb', gbcc), ('rf', rfcc)],


    final_estimator=LogisticRegression(class_weight='balanced',
                                       C=60,
                                       solver="lbfgs",
                                       max_iter=1000,

                                       ),

    # uses cross-val(cv) predictions to train meta-learner
    cv = 4


)

stacking.fit(X_train, y_train)
print(f"GBRFC Accuracy = {stacking.score(X_test, y_test):.3f}")
        #^ gradient boosted random forest
C = stacking.score(X_test, y_test)

GBRFC Accuracy = 0.903


In [195]:
C

0.9032258064516129

In [204]:
Difference = H - C
# difference between control and hyperparameter
Difference

0.048387096774193616